In [ ]:
!pip install gmail


  Preparing metadata (setup.py) ... done
  Created wheel for gmail: filename=gmail-0.6.3-py3-none-any.whl size=10756 sha256=e5ac0ed7793edaded865dff19a000005ce8e7122bcf6d9220e598fe887df47d8
  Stored in directory: /root/.cache/pip/wheels/f4/a1/3c/7bde7746cc4429112f601dfc00c9f1accd0880f9ce11dcc918
Successfully built gmail


In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
!pip install pyngrok
!pip install gspread oauth2client requests

In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
from google.auth import default
import gspread
import asyncio
import nest_asyncio
import time
from flask import Flask , request , jsonify
from pyngrok import ngrok





def send_the_email(rec_email,sub,main_message):
    sender_email = "sachinbhati.299792458@gmail.com"
    receiver_email = rec_email

    app_paasword = "your-app-password"
    messsage = MIMEMultipart()
    messsage["From"] = sender_email
    messsage["To"] = receiver_email
    messsage["Subject"] = sub
    body = main_message
    messsage.attach(MIMEText(body,"plain"))


    filename = "resume.docx"   

    with open(filename, "rb") as attachment:

        part = MIMEBase("application", "octet-stream")
        part.set_payload(attachment.read())

 
    encoders.encode_base64(part)

    part.add_header(
        "Content-Disposition",
        f"attachment; filename={filename}",
    )

    messsage.attach(part)
    server = smtplib.SMTP("smtp.gmail.com",587)
    server.starttls()
    server.login(sender_email,app_paasword)
    server.sendmail(sender_email,receiver_email,messsage.as_string())
    server.quit()
    print("success")

creds, _ = default()
gs = gspread.authorize(creds)
sh = gs.open("hello automation")
ws = sh.worksheet("Sheet4")


nest_asyncio.apply()
app = Flask(__name__)
NGROK_TOKEN = "ngrok-auth-token" # Get this from dashboard.ngrok.com



async def start_batch(target_of_email):
    i = 2
    sent = 0
    target = target_of_email
    print(target)
    while (True):
      row = ws.row_values(i)
      if row[3] == "sent":
        i+=1
        continue
      send_the_email(row[0],row[1],row[2])
      ws.update_cell(i,4,"sent")
      sent += 1
      if sent == target :
        break
      if i >1500:
        break
      i+=1

@app.route('/trigger-colab',methods=['POST'])
def trigger():
    print("running")
    target = int(request.args.get("target",1))
    asyncio.run(start_batch(target))
    return jsonify({"status":"success"}),200

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000).public_url
print(f" * PUBLIC NGROK URL: {public_url}/trigger-colab")

if __name__ == '__main__':
    app.run(port=5000)


 * PUBLIC NGROK URL: https://81e6-136-116-161-191.ngrok-free.app/trigger-colab
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


running
2
success


INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:15:12] "POST /trigger-colab?target=2 HTTP/1.1" 200 -


success
